# Download historical data from Binance (USDM Futures)

In [1]:
import requests
import datetime
import time
import os
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor
print(os.getcwd())

/mnt/c/Users/alexs/Desktop/levbot


#### Sanity check

In [18]:
baseurl = "https://data.binance.vision/?prefix=data/futures/cm/monthly/klines/"
res = requests.head(baseurl)
print(res)

<Response [200]>


#### Generate filenames 

In [26]:
def coinDirURL(coin: str, timeframe: str) -> str:
    """
    Create director URL for the given coin and timeframe.
    :param coin: "BTC"
    :param timeframe: "1m"
    :return: url
    """
    baseurl = "https://data.binance.vision/data/futures/cm/monthly/klines/"
    return f"{baseurl + coin}USD_PERP/{timeframe}/"

def getAvailable(coin: str, timeframe: str) -> list:
    """
     Check available months, counting down from the current date
    :param coin: BTC
    :param timeframe: 1m
    :return: urls of files that can be downloaded
    """
    directory = coinDirURL(coin, timeframe)
    month = int(datetime.datetime.now().strftime("%m"))
    year = int(datetime.datetime.now().strftime("%Y"))
    
    fileprefix = f"{coin}USD_PERP-{timeframe}"
    
    result = []
    while True: # infinite loop until we break
        time.sleep(0.05)
        month -= 1
        # Loop back to december
        if month == 0:
            year -= 1
            month = 12
        
        # URL of the file we are checking, formatting month to have leading zeros
        fileurl = f"{directory+fileprefix}-{year}-{format(month, '02d')}.zip"
        
        # Send the request
        res = requests.head(fileurl)
        print(f"{year} - {month}, response {res.status_code}")
        
        # Check if it exists
        if res.status_code != 200:
            # Does not exist, no more data, return what we have
            return result
        else:
            # Append to results
            result.append(fileurl)
            
        


### Download the files

In [28]:
def dwnld(url, timeframe):
    # Create filename from url
    coin = url.split("/")[-1][0:3]
    suffix = timeframe + url[-12:]
    filename = coin + "/zipped/" + suffix
    
    try:
        os.makedirs(coin + "/zipped/")
    except FileExistsError:
        # directory already exists
        pass

    # request!
    response = requests.get(url)
    
    if response.status_code != 200:
        print("Error downloading "+"filename")
    
    try:
        with open(filename, mode="wb") as file:
            file.write(response.content)
            file.close()
        print(f"Downloaded file {filename}")
    except Exception as e:
        print(e)

### Unzip the files 

In [29]:
def unzipall(coins):
    import shutil
    for coin in coins:
        for file in tqdm(os.listdir(coin+"/zipped")):
            finaldirectory = coin + "/csv/" +file.split("-")[0]
            print(finaldirectory)
            try:
                os.makedirs(finaldirectory)
            except FileExistsError:
                # directory already exists
                pass
            if file.endswith(".zip"):
                shutil.unpack_archive(coin+"/zipped/" +file, finaldirectory)


def downloadZipsForTimeframe(coin, timeframe):
    try:
        filenames = getAvailable(coin, timeframe)
        timeframes = []
        for filename in filenames:
            timeframes.append(timeframe)
        
        with ThreadPoolExecutor() as executor:
            executor.map(dwnld, filenames, timeframes)
    except Exception as e:
        print(e)


def downloadhistorical(coins, timeframes):
    from concurrent.futures import ThreadPoolExecutor
    
    with ThreadPoolExecutor() as executor:
        for coin in coins:
            # Create an equivalent array of just the coin for the map
            coinplural = []
            for timeframe in timeframes:
                coinplural.append(coin)
            executor.map(downloadZipsForTimeframe, coinplural, timeframes)
    unzipall(coins)


In [1]:
import Data.BinanceDownloader
from tqdm.auto import tqdm
baseurl = "https://data.binance.vision/data/futures/cm/monthly/"
dwnlder = Data.BinanceDownloader.Downloader(baseurl, "Data/test")

In [4]:
pbar = tqdm(desc=f"Checking available data", unit=" response")
av = dwnlder.getAvailable(pbar)
pbar.set_description("Downloading")
dwnlder.downloadAll(av,pbar)
pbar.close()

Checking available data: 0 response [00:00, ? response/s]

saved BTCUSD_PERP-1d-2024-09.zip
saved BTCUSD_PERP-1d-2024-11.zip
saved BTCUSD_PERP-1d-2024-10.zip
saved BTCUSD_PERP-1d-2024-05.zip
saved BTCUSD_PERP-1d-2024-08.zip
saved BTCUSD_PERP-1d-2024-12.zip
saved BTCUSD_PERP-1d-2024-06.zip
saved BTCUSD_PERP-1d-2023-04.zip
saved BTCUSD_PERP-1d-2024-01.zip
saved BTCUSD_PERP-1d-2023-12.zip
saved BTCUSD_PERP-1d-2024-02.zip
saved BTCUSD_PERP-1d-2023-09.zip
saved BTCUSD_PERP-1d-2023-08.zip
saved BTCUSD_PERP-1d-2023-05.zip
saved BTCUSD_PERP-1d-2023-10.zip
saved BTCUSD_PERP-1d-2024-03.zip
saved BTCUSD_PERP-1d-2023-06.zip
saved BTCUSD_PERP-1d-2023-07.zip
saved BTCUSD_PERP-1d-2024-07.zip
saved BTCUSD_PERP-1d-2023-11.zip
saved BTCUSD_PERP-1d-2024-04.zip
saved BTCUSD_PERP-1d-2022-09.zip
saved BTCUSD_PERP-1d-2022-10.zip
saved BTCUSD_PERP-1d-2022-11.zip
saved BTCUSD_PERP-1d-2022-05.zip
saved BTCUSD_PERP-1d-2022-03.zip
saved BTCUSD_PERP-1d-2022-04.zip
saved BTCUSD_PERP-1d-2022-02.zip
saved BTCUSD_PERP-1d-2021-12.zip
saved BTCUSD_PERP-1d-2021-10.zip
saved BTCU

In [3]:
import Data.BinanceDownloader
from tqdm.auto import tqdm
baseurl = "https://data.binance.vision/data/futures/cm/monthly/"
coindownloader = Data.BinanceDownloader.CoinDownloader(baseurl, coin ="AAVEUSD_PERP", savefolder = "Data")
#coindownloader.downloadTimeFrame(datatype = "klines", timeframe = "1m")

In [2]:
coins = ["BTCUSD_PERP", "ETHUSD_PERP", "ADAUSD_PERP", "SOLUSD_PERP"]#, "XRPUSD_PERP", "BNBUSD_PERP", "DOGEUSD_PERP", "LINKUSD_PERP", "XLMUSD_PERP", "TRXUSD_PERP", "AVAXUSD_PERP", "DOTUSD_PERP"]

datatypes = ["klines"]

timeframes = ["1m", "3m", "5m", "15m", "30m", "1h", "6h", "12h", "1d"]

Data.BinanceDownloader.bulkCoinDatatypeTimeframe(coins, datatypes, timeframes)

Bulk Downloading!:   0%|          | 0/4 [00:00<?, ?it/s]

15m: Checking available data: 0 response [00:00, ? response/s]

6h: Checking available data: 0 response [00:00, ? response/s]

30m: Checking available data: 0 response [00:00, ? response/s]

1m: Checking available data: 0 response [00:00, ? response/s]

...

5m: Checking available data: 0 response [00:00, ? response/s]

1d: Checking available data: 0 response [00:00, ? response/s]

5m: Checking available data: 0 response [00:00, ? response/s]

1h: Checking available data: 0 response [00:00, ? response/s]

1d: Checking available data: 0 response [00:00, ? response/s]